In [27]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def perform_eda():
    """
    Reads all labeled stock data, combines it, and performs a focused exploratory
    data analysis by generating and saving key visualizations that highlight
    the distribution of labels across various technical indicators, including
    the new "EndOfConsolidation" label.
    """
    try:
        # Define paths
        labeled_data_folder = "LabeledData"
        output_folder = "EDA_Plots"

        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)

        # Set a consistent plot style and larger font sizes for better readability
        sns.set_style('whitegrid')
        plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16})

        # List to hold all dataframes
        all_dfs = []
        
        # Loop through all files in the LabeledData folder
        for filename in os.listdir(labeled_data_folder):
            if filename.endswith(".csv"):
                file_path = os.path.join(labeled_data_folder, filename)
                print(f"Reading {filename}...")
                
                # Read the CSV file into a dataframe
                df = pd.read_csv(file_path)
                
                # Add a 'Symbol' column from the filename
                df['Symbol'] = os.path.splitext(filename)[0].split('.')[0]
                
                all_dfs.append(df)

        if not all_dfs:
            print("No CSV files found in the LabeledData folder.")
            return

        # Concatenate all dataframes into a single one
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print("Successfully combined all labeled data into a single dataframe.")
        
        # Drop the 'Unnamed: 0' column if it exists
        if 'Unnamed: 0' in combined_df.columns:
            combined_df = combined_df.drop('Unnamed: 0', axis=1)
        
        # Ensure the 'Label' column is a categorical type with a consistent order
        label_order = ['NotImportant', 'Consolidation', 'EndOfConsolidation', 'Breakout']
        combined_df['Label'] = pd.Categorical(combined_df['Label'], categories=label_order, ordered=True)

        print("\n--- Generating Plots Focused on Label Distributions ---")

        # 1. Bar Plot: Overall Distribution of Labels
        plt.figure(figsize=(10, 6))
        sns.countplot(x='Label', data=combined_df)
        plt.title('Distribution of Labels Across All Stocks')
        plt.xlabel('Label')
        plt.ylabel('Number of Occurrences')
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, 'label_distribution.png'), dpi=300)
        print("1. Saved 'label_distribution.png'")
        plt.close()

        # 2. Box Plot: Weekly Return by Label
        plt.figure(figsize=(12, 8))
        sns.boxplot(x='Label', y='Weekly_Return', data=combined_df)
        plt.title('Weekly Return Distribution by Label')
        plt.xlabel('Label')
        plt.ylabel('Weekly Return (%)')
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, 'weekly_return_by_label.png'), dpi=300)
        print("2. Saved 'weekly_return_by_label.png'")
        plt.close()

        # 3. Violin Plots for key indicators
        indicators_for_violin = ['RSI', 'ROC_10', 'Weekly_Return', 'Stochastic_%K', 'Normalized_Open', 'Volume_Spike_Ratio_5Wk']
        for indicator in indicators_for_violin:
            plt.figure(figsize=(12, 8))
            sns.violinplot(x='Label', y=indicator, data=combined_df)
            plt.title(f'Distribution of {indicator} by Label')
            plt.xlabel('Label')
            plt.ylabel(indicator)
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'{indicator}_violin.png'), dpi=300)
            print(f"3. Saved '{indicator}_violin.png'")
            plt.close()
        
        # 4. Histograms for key indicators, separated by label
        indicators_for_hist = ['Normalized_Open', 'Volume_Spike_Ratio_5Wk']
        for indicator in indicators_for_hist:
            plt.figure(figsize=(12, 8))
            
            # Use all labels for plotting
            data_to_plot = combined_df
            plot_title = f'Distribution of {indicator} by Label'

            sns.histplot(data=data_to_plot, x=indicator, hue='Label', multiple='stack', kde=True, bins=50)
            
            plt.title(plot_title)
            plt.xlabel(indicator)
            plt.ylabel('Frequency')
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'{indicator}_histogram.png'), dpi=300)
            print(f"4. Saved '{indicator}_histogram.png'")
            plt.close()

        # 5. Pairplot for a comprehensive view
        key_features = ['Weekly_Return', 'RSI', 'Stochastic_%K']
        # Filter to include all relevant labels
        pairplot_df = combined_df[combined_df['Label'].isin(['Consolidation', 'EndOfConsolidation', 'Breakout'])][['Label'] + key_features].dropna()
        
        if not pairplot_df.empty:
            print("5. Generating comprehensive pairplot. This may take a moment...")
            sns.pairplot(pairplot_df, hue='Label', diag_kind='kde')
            plt.suptitle('Pairwise Relationships and Distributions (Critical Labels)', y=1.02, fontsize=16)
            plt.savefig(os.path.join(output_folder, 'pairplot.png'), dpi=300)
            print("5. Saved 'pairplot.png'")
            plt.close()
        else:
            print("Not enough data to generate pairplot. Skipping.")
            
        # 6. Bar Plot: Breakouts and Consolidations per Stock
        print("6. Generating bar plot of breakouts and consolidations per stock...")
        
        filtered_df = combined_df[combined_df['Label'].isin(['Breakout', 'Consolidation', 'EndOfConsolidation'])].copy()
        
        if not filtered_df.empty:
            label_counts = filtered_df.groupby(['Symbol', 'Label']).size().reset_index(name='Count')
            
            plt.figure(figsize=(15, 8))
            sns.barplot(x='Symbol', y='Count', hue='Label', data=label_counts)
            plt.title('Number of Breakouts and Consolidations per Stock')
            plt.xlabel('Stock Symbol')
            plt.ylabel('Number of Occurrences')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, 'breakouts_consolidations_per_stock.png'), dpi=300)
            print("6. Saved 'breakouts_consolidations_per_stock.png'")
            plt.close()
        else:
            print("No 'Breakout', 'Consolidation' or 'EndOfConsolidation' labels found. Skipping plot.")
            
        # 7. Time-series plot with Bollinger Z-Score for all stocks
        print("\n7. Generating a time-series plot for Bollinger Z-Score for each stock...")
        
        # Get a list of all unique symbols
        unique_symbols = combined_df['Symbol'].unique()
        
        for symbol in unique_symbols:
            sample_df = combined_df[combined_df['Symbol'] == symbol].copy()
            
            # Check for the required columns
            if 'Bollinger_Z_Score' not in sample_df.columns or 'Date' not in sample_df.columns:
                print(f"Warning: The 'Bollinger_Z_Score' or 'Date' column was not found for symbol {symbol}. Skipping Z-Score plot for this stock.")
                continue

            # Ensure the date column is in datetime format and sort
            sample_df['Date'] = pd.to_datetime(sample_df['Date'])
            sample_df = sample_df.sort_values('Date').reset_index(drop=True)
            
            if not sample_df.empty:
                plt.figure(figsize=(15, 8))
                ax = plt.gca()
                
                # Plot the Bollinger Z-Score
                ax.plot(sample_df['Date'], sample_df['Bollinger_Z_Score'], label='Bollinger Z-Score', color='dodgerblue')
                
                # Add horizontal lines for visual reference at +2 and -2
                ax.axhline(y=2, color='red', linestyle='--', label='+2 Std Dev')
                ax.axhline(y=-2, color='red', linestyle='--', label='-2 Std Dev')
                ax.axhline(y=0, color='gray', linestyle='-', label='SMA (0)')
                
                # Highlight breakout points with a red scatter plot
                breakouts = sample_df[sample_df['Label'] == 'Breakout']
                ax.scatter(breakouts['Date'], breakouts['Bollinger_Z_Score'], color='red', marker='^', s=100, label='Breakout Event', zorder=5)
                
                # Highlight consolidation points with a green scatter plot
                consolidations = sample_df[sample_df['Label'] == 'Consolidation']
                ax.scatter(consolidations['Date'], consolidations['Bollinger_Z_Score'], color='green', marker='o', s=100, label='Consolidation Event', zorder=5)

                # Highlight EndOfConsolidation points
                end_of_consolidations = sample_df[sample_df['Label'] == 'EndOfConsolidation']
                ax.scatter(end_of_consolidations['Date'], end_of_consolidations['Bollinger_Z_Score'], color='purple', marker='v', s=100, label='EndOfConsolidation Event', zorder=5)

                ax.set_title(f'Bollinger Z-Score and Labeled Events for {symbol}', fontsize=16)
                ax.set_xlabel('Date')
                ax.set_ylabel('Bollinger Z-Score')
                ax.legend()
                plt.tight_layout()
                
                # Save the plot with the symbol in the filename
                plt.savefig(os.path.join(output_folder, f'bollinger_z_score_{symbol}.png'), dpi=300)
                print(f"Saved 'bollinger_z_score_{symbol}.png'")
                plt.close()
            else:
                print(f"Could not generate Bollinger Z-Score plot for {symbol}. Data might be missing or insufficient.")
            
        print("\nFocused EDA is complete. Please check the 'EDA_Plots' folder for the new graphs.")

    except FileNotFoundError as e:
        print(f"Error: The folder or file was not found. Please ensure the '{labeled_data_folder}' folder exists with the CSV files. Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during EDA: {e}")

# Run the main function
if __name__ == "__main__":
    perform_eda()


Reading MM.csv...
Reading ICICIBANK.csv...
Reading MARUTI.csv...
Reading ASIANPAINT.csv...
Reading APOLLOHOSP.csv...
Reading HDFCBANK.csv...
Reading ADANIENT.csv...
Reading HEROMOTOCO.csv...
Reading SBIN.csv...
Reading BAJAJFINSV.csv...
Reading BAJFINANCE.csv...
Successfully combined all labeled data into a single dataframe.

--- Generating Plots Focused on Label Distributions ---
1. Saved 'label_distribution.png'
2. Saved 'weekly_return_by_label.png'
3. Saved 'RSI_violin.png'
3. Saved 'ROC_10_violin.png'
3. Saved 'Weekly_Return_violin.png'
3. Saved 'Stochastic_%K_violin.png'
3. Saved 'Normalized_Open_violin.png'
3. Saved 'Volume_Spike_Ratio_5Wk_violin.png'
4. Saved 'Normalized_Open_histogram.png'
4. Saved 'Volume_Spike_Ratio_5Wk_histogram.png'
5. Generating comprehensive pairplot. This may take a moment...
5. Saved 'pairplot.png'
6. Generating bar plot of breakouts and consolidations per stock...
6. Saved 'breakouts_consolidations_per_stock.png'

7. Generating a time-series plot for B